# 📡 Aurora Forecasts — Data Retrieval

> **Author:** `<!-- your name -->`  · **Date:** `<!-- date -->`  · **Version:** 1.0

---

## Abstract

This notebook orchestrates the retrieval of Aurora Energy Research forecast scenario data via the Aurora HTTP API. It downloads **technology** and **system** scenario data at multiple time resolutions — **yearly (1y)**, **monthly (1m)**, and **hourly (1h, WIP)** — across configured European regions, persisting all results as Parquet files for downstream consumption.

Running this notebook is **Step 1** in the Aurora data pipeline. Downstream notebooks read the Parquet files produced here for aggregation, visualisation, and export to Excel.

---

## 🗺️ Notebook Map

| Step | Section | Description |
|---|---|---|
| 1 | Imports | Load `pandas` and the `AuroraAPI` retrieval client |
| 2 | Data Paths | Define Parquet and scenario-registry file paths per resolution |
| 3 | API Config | Read API token and country list from `api_params.yaml` |
| 4 | Yearly API | Initialise client, inspect available currencies, fetch yearly scenarios |
| 5 | Monthly API | Initialise client, fetch monthly scenarios |
| 6 | Hourly API | Initialise client — fetch step is currently a WIP |

> ⚙️ **Config source:** `config/aurora/api_params.yaml`
> 💾 **Output location:** `notebooks/data/aurora/` (Parquet files)

## 📦 1 · Imports & Environment Setup

Load standard data-manipulation libraries and the project-specific Aurora API client.

**Why here?** Centralising all imports at the top makes dependencies explicit and ensures the notebook fails fast if a package is missing.

| | |
|---|---|
| **Output** | `pd` (pandas) alias; `AuroraAPI` class ready for instantiation |

In [10]:
# Standard data-manipulation library — used for DataFrame inspection and type annotations
import pandas as pd

### Project Package Import

Import the `AuroraAPI` retrieval client from the `aurora_forecasts` package (installed in editable mode as part of this monorepo).

> 💡 **Tip:** If you see an `ImportError` here, run `pip install -e packages/aurora_forecasts` from the repository root.

In [11]:
# AuroraAPI: high-level client wrapping the Aurora Energy Research HTTP API.
# Handles authentication, incremental scenario registry management,
# chunked data retrieval, and persistence to Parquet.
from aurora_forecasts.retrieval_helper.retrieval_helper import AuroraAPI

## 🗂️ 2 · Configure Data Paths

Define file paths for scenario-component registries (JSON) and Parquet output databases for each time resolution.

**Why separate registries?** Each resolution (1y, 1m, 1h) maintains its own registry of already-downloaded scenario components to avoid redundant API calls on subsequent runs — only new or updated components are fetched.

| | |
|---|---|
| **Input** | Repository-relative path strings |
| **Output** | Path variables consumed by all `AuroraAPI` constructors below |

In [12]:
# --- Scenario-component registry paths (JSON) ---
# Each registry tracks which scenario components have already been fetched,
# preventing duplicate downloads on incremental re-runs.
aurora_registry_monthly_path = '../../../data/aurora/aurora_scenario_components_registry_1m.json'
aurora_registry_yearly_path  = '../../../data/aurora/aurora_scenario_components_registry_1y.json'
aurora_registry_hourly_path  = '../../../data/aurora/aurora_scenario_components_registry_1h.json'

# --- Parquet output paths — Yearly resolution ---
technology_path_yearly = '../../../data/aurora/aurora2_technology_scenarios_ES_default_currency_1y.parquet'
system_path_yearly     = '../../../data/aurora/aurora2_system_scenarios_ES_default_currency_1y.parquet'

# --- Parquet output paths — Monthly resolution ---
technology_path_monthly = '../../../data/aurora/aurora_technology_scenarios_ES_default_currency_1m.parquet'
system_path_monthly     = '../../../data/aurora/aurora_system_scenarios_ES_default_currency_1m.parquet'

# --- Parquet output paths — Hourly resolution ---
technology_path_hourly = '../../../data/aurora/aurora_technology_scenarios_ES_default_currency_1h.parquet'
system_path_hourly     = '../../../data/aurora/aurora_system_scenarios_ES_default_currency_1h.parquet'

# Path to the YAML config file containing the API token and country code list
api_config = '../../../config/aurora/api_params.yaml'

## 🔑 3 · Load API Configuration

Read the Aurora API token and target country codes from the YAML config file.

**Why YAML?** Externalising credentials to a config file keeps secrets out of version control and makes it easy to rotate the API token without touching notebook code.

| | |
|---|---|
| **Input** | `config/aurora/api_params.yaml` |
| **Output** | `aurora_token` (str), `country_code_list` (list[str]) |

In [13]:
import yaml

# Load the full YAML config block for the Aurora API
with open(api_config, 'r') as file:
    api_params = yaml.safe_load(file)

# Extract only the two fields needed for API authentication and geography targeting
aurora_token      = api_params['aurora_token']       # Bearer token for Aurora HTTP API
country_code_list = api_params['country_code_list']  # e.g. ['ES', 'DE', 'FR']

## 📅 4 · Yearly Resolution API — Initialise & Inspect

Instantiate an `AuroraAPI` client configured for yearly (1y) data, then inspect the available currency options.

**Why inspect currencies?** Aurora delivers prices in the currency specified at download time. Reviewing the currency catalogue confirms which base-year real values are available before committing to a full scenario pull.

| | |
|---|---|
| **Input** | `aurora_token`, `country_code_list`, yearly Parquet / registry paths |
| **Output** | `aurora_yr` client instance; `df_currencies` DataFrame displayed for reference |

In [14]:
# Instantiate the yearly AuroraAPI client.
# Passing country_code_list here sets the default geography for all operations
# on this instance; it can be overridden per-call in update_scenario_database().
aurora_yr = AuroraAPI(
    aurora_token,
    scenario_comp_registry_path=aurora_registry_yearly_path,
    technology_db_path=technology_path_yearly,
    country_code_list=country_code_list,
    system_db_path=system_path_yearly
    )

## ⬇️ 5 · Fetch Yearly Forecast Scenarios

Download and persist all yearly scenario components for the configured countries.

**How it works:** `update_scenario_database()` compares available scenario components from the API against the local registry. Only missing components are fetched and appended to the Parquet store — making incremental runs efficient.

| | |
|---|---|
| **Input** | `aurora_yr` client (already configured with paths and country list) |
| **Output** | Updated `technology_path_yearly` and `system_path_yearly` Parquet files |

> ⚠️ **Note:** This cell may take several minutes on first run when the local registry is empty.

In [15]:
# Trigger incremental yearly scenario download.
# Default resolution is '1y'; country list was set at construction time.
# Only scenario components absent from the local registry are fetched.
aurora_yr.update_scenario_database()

Retrieving data for scenario: Poland Nov 2020 (High)
No data retrieved for scenario: Poland Nov 2020 (High)
Retrieving data for scenario: Poland Nov 2020 (Low)
No data retrieved for scenario: Poland Nov 2020 (Low)
Scenario Iberia October 2020 (Central) for region prt already in registry, skipping retrieval.
Scenario Iberia October 2020 (High) for region prt already in registry, skipping retrieval.
Scenario Iberia October 2020 (Low) for region prt already in registry, skipping retrieval.
Scenario Iberia October 2020 (High) for region esp already in registry, skipping retrieval.
Scenario Iberia October 2020 (Low) for region esp already in registry, skipping retrieval.
Scenario Ireland Nov 2020 (Low) for region irx already in registry, skipping retrieval.
Scenario Ireland Nov 2020 (High) for region irx already in registry, skipping retrieval.
Scenario France Oct 2020 (Central) for region fra already in registry, skipping retrieval.
Scenario France Oct 2020 (Low) for region fra already in 

## 📆 6 · Monthly Resolution API — Initialise

Instantiate a second `AuroraAPI` client configured for monthly (1m) scenario retrieval.

| | |
|---|---|
| **Input** | `aurora_token`; monthly Parquet / registry paths |
| **Output** | `aurora` client instance ready for monthly fetch |

In [16]:
# Instantiate a second AuroraAPI client for monthly resolution.
# country_code_list is not set here — it will be passed explicitly to update_scenario_database().
aurora = AuroraAPI(
    aurora_token,
    scenario_comp_registry_path=aurora_registry_monthly_path,
    technology_db_path=technology_path_monthly,
    system_db_path=system_path_monthly
    )

## ⬇️ 7 · Fetch Monthly Forecast Scenarios

Download and persist all monthly scenario components for the configured countries.

| | |
|---|---|
| **Input** | `aurora` client; `country_code_list`; `resolution='1m'` |
| **Output** | Updated monthly Parquet files (see path note in §6) |

> ⚠️ **Note:** Monthly granularity produces significantly more data than yearly — first run may be slow.

In [9]:
# Fetch monthly scenario data; resolution='1m' overrides the default '1y'.
# country_code_list is passed explicitly because it was not set at construction time.
aurora.update_scenario_database(
    resolution='1m',
    country_code_list=country_code_list
)

Retrieving data for scenario: Poland Nov 2020 (High)
Forecast download failed: 404
No data retrieved for scenario: Poland Nov 2020 (High)
Retrieving data for scenario: Poland Nov 2020 (Low)
Forecast download failed: 404
No data retrieved for scenario: Poland Nov 2020 (Low)
Retrieving data for scenario: Iberia October 2020 (Central)
Forecast download failed: 404
No data retrieved for scenario: Iberia October 2020 (Central)
Retrieving data for scenario: Iberia October 2020 (High)
Forecast download failed: 404
No data retrieved for scenario: Iberia October 2020 (High)
Retrieving data for scenario: Iberia October 2020 (Low)
Forecast download failed: 404
No data retrieved for scenario: Iberia October 2020 (Low)
Retrieving data for scenario: Iberia October 2020 (High)
Forecast download failed: 404
No data retrieved for scenario: Iberia October 2020 (High)
Retrieving data for scenario: Iberia October 2020 (Low)
Forecast download failed: 404
No data retrieved for scenario: Iberia October 2020 

## ⏱️ 8 · Hourly Resolution API — Initialise & Single Forecast Retrieval

Instantiate the `AuroraAPI` client for hourly (1h) scenario retrieval, then fetch a single forecast for ad-hoc inspection.

**Status:** The bulk `update_scenario_database` call for hourly resolution is commented out — full batch download is a work in progress. `retrieve_single_forecast` is used instead to pull one specific scenario component for validation.

| | |
|---|---|
| **Input** | `aurora_token`, hourly Parquet / registry paths; scenario hash + parameters |
| **Output** | `aurora_h` client instance; `df` DataFrame with forecast values for the specified scenario |

In [ ]:
# Instantiate the hourly AuroraAPI client with hourly-specific Parquet and registry paths.
aurora_h = AuroraAPI(
    aurora_token,
    scenario_comp_registry_path=aurora_registry_hourly_path,
    technology_db_path=technology_path_hourly,
    system_db_path=system_path_hourly
)


: 

---

## 🏁 Summary & Next Steps

### What We Did
- Imported `pandas` and the `AuroraAPI` retrieval client from the `aurora_forecasts` package.
- Defined all Parquet output paths and scenario-registry paths for yearly, monthly, and hourly resolutions.
- Loaded the Aurora API token and target country list from `config/aurora/api_params.yaml`.
- Initialised a yearly `AuroraAPI` client, inspected the available currency catalogue, and fetched yearly forecast scenarios.
- Initialised a monthly `AuroraAPI` client and fetched monthly forecast scenarios.
- Initialised an hourly `AuroraAPI` client and retrieved a single forecast for ad-hoc validation (bulk hourly fetch pending).

### Output Files Produced

| Resolution | Technology Parquet | System Parquet |
|---|---|---|
| Yearly (1y) | `aurora2_technology_scenarios_ES_default_currency_1y.parquet` | `aurora2_system_scenarios_ES_default_currency_1y.parquet` |
| Monthly (1m) | `aurora_technology_scenarios_ES_default_currency_1m.parquet` | `aurora_system_scenarios_ES_default_currency_1m.parquet` |
| Hourly (1h) | `aurora_technology_scenarios_ES_default_currency_1h.parquet` *(WIP)* | `aurora_system_scenarios_ES_default_currency_1h.parquet` *(WIP)* |

### Suggested Next Steps
1. **Complete hourly fetch** — implement and test `update_scenario_database(resolution='1h', ...)` in `AuroraAPI`.
2. **Proceed to downstream notebook** — load the Parquet files produced here for data cleaning, aggregation, and visualisation.

---
*Generated documentation · feel free to edit and expand.*